# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')
    
    # directly download dataset
    ! mkdir -p 'prepared'
    ! wget -nv -P 'datasets' https://raw.githubusercontent.com/mgmalheiros/vision/master/datasets/1-coins-small-image.zip
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL'))

# Image conversion
1. select **10** random images from the **small** dataset
2. convert images to gray
3. save images to a **prepared** dataset, in a **deterministic ZIP** file

In [ ]:
import io
import imageio.v3 as iio
import random
import skimage
import zipfile

In [ ]:
random.seed(0) # ensure reproducibility

date_time = (1980, 1, 1, 0, 0, 0) # ensure all zip entries have the same time

original = base_folder / 'datasets' / '1-coins-small-image.zip'
prepared = base_folder / 'prepared' / '1-coins-tiny-image.zip'

with zipfile.ZipFile(original, 'r') as src, \
     zipfile.ZipFile(prepared, 'w') as dst:
    namelist = [n for n in src.namelist() if n.endswith('.jpg')]
    namelist = random.sample(namelist, k=10)

    for input_name in sorted(namelist): # ensure zip entries are in sorted order
        data = io.BytesIO(src.read(input_name))
        image = skimage.io.imread(data, as_gray=True)
        image = skimage.util.img_as_ubyte(image)

        output_name = pathlib.Path(input_name).with_suffix('.jpg').name
        
        # save image (for debugging)
        #skimage.io.imsave(output_name, image)
        #print('saved', output_name)

        data = io.BytesIO()
        iio.imwrite(data, image, extension='.jpg')
        dst.writestr(zipfile.ZipInfo(output_name, date_time=date_time), data.getbuffer())
        print('wrote', output_name)

# Testing

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image)
plt.show()